# COCO15 · cup

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/integration/Klygo_COCO15_08_cup.ipynb)


This notebook evaluates **cup** (COCO15 class `8`, original COCO class `41`) on real COCO128 photographs. It exercises Klygo's `files`, `config`, `archive`, `datasets`, `media`, `preprocessing`, `models`, `outputs`, `postprocessing`, `visual`, `validators`, and `utils` in one workflow.

Sign in to Google Colab, choose **Runtime → Change runtime type → GPU**, and upload `klygo_coco15_bundle.zip` once to `MyDrive/Klygo/`. The notebook asks you to authorize Drive access. A GPU speeds up inference; it falls back to CPU if none is assigned. The bundle contains this checkout's Klygo source and the 15-class dataset, so its behavior does not depend on unpushed Git changes.

For a quick run, this notebook uses up to four positive and two negative images. Set `KLYGO_FULL_DATASET=1` before the sampling cell to evaluate all 128 images. COCO128 is a pipeline test subset, not an independent model benchmark.


In [ ]:
import os, sys, tempfile, zipfile, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    default_bundle = Path("/content/drive/MyDrive/Klygo/klygo_coco15_bundle.zip")
    WORK_ROOT = Path("/content/klygo_coco15")
    os.environ.setdefault("HF_HOME", "/content/drive/MyDrive/Klygo/hf_cache")
else:
    default_bundle = Path("artifacts/integration/bundles/klygo_coco15_bundle.zip")
    WORK_ROOT = Path(tempfile.gettempdir()) / "klygo_coco15_notebook"
    os.environ.setdefault("HF_HUB_OFFLINE", "1")
    os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

BUNDLE = Path(os.getenv("KLYGO_COCO15_BUNDLE", str(default_bundle)))
if not BUNDLE.is_file():
    raise FileNotFoundError(f"Upload the bundle to Drive or set KLYGO_COCO15_BUNDLE: {BUNDLE}")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as bundle:
    for name in bundle.namelist():
        parts = Path(name).parts
        if Path(name).is_absolute() or ".." in parts:
            raise ValueError(f"Unsafe bundle path: {name}")
    bundle.extractall(WORK_ROOT)
PROJECT_ROOT = WORK_ROOT / "project"
DATASET_ROOT = WORK_ROOT / "dataset"
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT), "transformers>=4.50,<6"])
print("Bundle:", BUNDLE)
print("Dataset:", DATASET_ROOT)


In [ ]:
import json
import torch
from klygo import archive, config, datasets, files, media, models, outputs
from klygo import postprocessing, preprocessing, utils, validators, visual

WORK_ROOT = files.path(WORK_ROOT)
DATASET_ROOT = files.path(DATASET_ROOT)

TARGET_CLASS = 'cup'
TARGET_ID = 8
validators.validate_type(TARGET_CLASS, str, "TARGET_CLASS")
manifest = files.load(files.join(DATASET_ROOT, "manifest.json"), verbose=False)
metadata = config.load(files.join(DATASET_ROOT, "data.yaml"), verbose=False)
dataset_info = datasets.info(DATASET_ROOT)
assert len(metadata.names) == 15 and metadata.names[TARGET_ID] == TARGET_CLASS
assert dataset_info["total_images"] == 128
print("Class:", TARGET_CLASS)
print("Labels:", manifest["classes"][TARGET_ID]["boxes"])
print("Images containing class:", manifest["classes"][TARGET_ID]["positive_images"])
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not assigned; using CPU")


In [ ]:
ground_truth = datasets.load(DATASET_ROOT, format="yolo", verbose=False)
truth_by_id = {files.stem(frame.config["relative_path"]): frame for frame in ground_truth}
positive_ids = sorted(image_id for image_id, frame in truth_by_id.items()
                      if any(box.label == TARGET_CLASS for box in frame.objects))
negative_ids = sorted(image_id for image_id, frame in truth_by_id.items()
                      if all(box.label != TARGET_CLASS for box in frame.objects))
full_dataset = os.getenv("KLYGO_FULL_DATASET") == "1"
max_positives = int(os.getenv("KLYGO_MAX_POSITIVES", "4"))
max_negatives = int(os.getenv("KLYGO_MAX_NEGATIVES", "2"))
selected_ids = sorted(truth_by_id) if full_dataset else positive_ids[:max_positives] + negative_ids[:max_negatives]
assert selected_ids and positive_ids
print("Evaluation images:", len(selected_ids), "positive:", sum(i in positive_ids for i in selected_ids))


In [ ]:
model = models.load("grounding-dino-tiny", size={"shortest_edge": 256, "longest_edge": 384})
if torch.cuda.is_available():
    model.model.to("cuda")
print("Registered model:", model.model_id)
print("Execution device:", next(model.model.parameters()).device)


In [ ]:
def score_image(truth, prediction):
    actual = [box for box in truth.objects if box.label == TARGET_CLASS]
    remaining = set(range(len(actual)))
    tp = fp = 0
    for box in sorted(prediction.objects, key=lambda item: item.score, reverse=True):
        if box.score < 0.25:
            continue
        candidates = [(postprocessing.box_iou(box, actual[index]), index)
                      for index in remaining]
        overlap, index = max(candidates, default=(0.0, -1))
        if overlap >= 0.5:
            tp += 1
            remaining.remove(index)
        else:
            fp += 1
    return tp, fp, len(remaining)

predictions, rows = {}, []
for image_id in selected_ids:
    image_path = files.join(DATASET_ROOT, "images", f"{image_id}.jpg")
    lazy_image = media.load(image_path, verbose=False)[0]
    prepared = preprocessing.auto_orient()(lazy_image)
    result = model.predict(prepared, prompt=[TARGET_CLASS], verbose=False)[0]
    assert isinstance(result, outputs.detect.Detection)
    filtered = result.transform(postprocessing.threshold(0.25), postprocessing.nms(0.5))
    filtered.transform(postprocessing.select(labels=[TARGET_CLASS]))
    tp, fp, fn = score_image(truth_by_id[image_id], filtered)
    predictions[image_id] = filtered
    rows.append({"image_id": image_id, "tp": tp, "fp": fp, "fn": fn,
                 "ground_truth": tp + fn, "predictions": tp + fp})

totals = {key: sum(row[key] for row in rows) for key in ("tp", "fp", "fn")}
totals["precision"] = totals["tp"] / max(1, totals["tp"] + totals["fp"])
totals["recall"] = totals["tp"] / max(1, totals["tp"] + totals["fn"])
print(json.dumps(totals, indent=2))


In [ ]:
RESULT_ROOT = files.join(
    "/content/drive/MyDrive/Klygo/results" if IN_COLAB
    else files.join(WORK_ROOT, "results"),
    TARGET_CLASS.replace(" ", "_"),
)
files.mkdir(RESULT_ROOT, parents=True, exist_ok=True)
preview_id = positive_ids[0]
preview = predictions.get(preview_id)
if preview is not None:
    original = media.load(files.join(DATASET_ROOT, "images", f"{preview_id}.jpg"), verbose=False)[0].to_pil()
    overlay = visual.draw_bboxes(original, preview.boxes, preview.labels, preview.scores)
    media.save(files.join(RESULT_ROOT, f"{preview_id}_prediction.jpg"), overlay, overwrite=True)
    display(overlay)
report = {"class_id": TARGET_ID, "class_name": TARGET_CLASS,
          "dataset_source_sha256": manifest["source_sha256"],
          "model_id": model.model_id, "device": str(next(model.model.parameters()).device),
          "confidence_threshold": 0.25, "iou_threshold": 0.5,
          "scope": "all 128 images" if full_dataset else f"up to {max_positives} positives and {max_negatives} negatives",
          "per_image": rows, "totals": totals}
files.save(files.join(RESULT_ROOT, "metrics.json"), report, overwrite=True, verbose=False)
archive_path = files.with_extension(RESULT_ROOT, ".zip")
archive.compress(RESULT_ROOT, archive_path, overwrite=True, verbose=False)
print("Report:", files.join(RESULT_ROOT, "metrics.json"))
print("Archive:", archive_path, utils.human_size(files.size(archive_path)))


## Review

Compare false positives and missed **cup** boxes in `metrics.json` and the saved overlay. The 15 notebooks share the same dataset and thresholds; they differ only in target class. The default sample is a smoke evaluation. For complete per-class coverage, set `KLYGO_FULL_DATASET=1` and rerun from the sampling cell.

Dataset: [Ultralytics COCO128](https://docs.ultralytics.com/datasets/detect/coco128). Grounding DINO weights are cached in your Drive after the first run. Drive authorization and GPU allocation are controlled by your Colab account.
